In [30]:
from typing import List, Any
import os
import weaviate
import json
import pandas as pd
from langchain_weaviate import WeaviateVectorStore
from weaviate.classes.query import Filter
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown
from dotenv import load_dotenv


In [31]:
load_dotenv('../.env.example/.env')

True

In [32]:
# weaviate Keys
WEAVIATE_URL = os.environ["WEAVIATE_URL"]
WEAVIATE_API_KEY = os.environ["WEAVIATE_API_KEY"]

In [33]:
weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=WEAVIATE_API_KEY,
)

In [34]:
weaviate_client.close()

In [35]:
weaviate_client.connect()

In [36]:
weaviate_client.is_live()

True

# weaviate collection

In [ ]:
Euro_Laws = weaviate_client.collections.use("Euro_Laws")

In [ ]:
eur_docs_hybrid = weaviate_client.collections.get("Euro_Laws_hybrid")

In [ ]:
# To get the full doc form Euro_Law_Documents collection
eur_docs = weaviate_client.collections.get("Euro_Law_Documents")

In [37]:
class SentenceTransformersEmbeddings:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # returns a list of embeddings for documents
        return self.model.encode(texts).tolist()

    def embed_query(self, text: str) -> List[float]:
        # returns a single embedding for a query
        return self.model.encode([text])[0].tolist()

In [38]:
embedding_model = SentenceTransformersEmbeddings('sentence-transformers/all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 502.47it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
quer_embed = embedding_model.embed_query("rape sentencing guidelines, punishment for rape, statutory penalties for rape, judicial discretion in rape cases")

In [16]:
response = Euro_Laws.query.near_vector(
    near_vector= v,
    limit=5
)

In [11]:
for obj in response.objects:
    print (obj.properties)

{'act_name': 'Directive 2012/29/EU of the European Parliament and of the Council of 25Ã\x82 October 2012 establishing minimum standards on the rights, support and protection of victims of crime, and replacing Council Framework Decision 2001/220/JHA', 'total_chunks': 31, 'status': 'In Force', 'legal_basis': '12010E082; 12010E294', 'eurovoc': 'crime against individuals; restorative justice; aid for victims; access to justice; AFSJ', 'act_type': 'Directive', 'celex': '32012L0029', 'chunk_number': 9, 'document_length': 78554, 'authors': 'European Parliament; European Council', 'subject_matter': 'criminal law;  justice;  European construction', 'cites': 'dec_framw/2002/475; 52012XX0209%2802%29; 32001R45; dec_framw/2008/977; 42000A0712%2801%29; 52010XG0504%2801%29; dec_framw/2009/948; 52009IP0098%2801%29; 32011L99; 32011L93; 52011IP0127; 32011G0628%2801%29; 32011L36', 'text': "competent authorities are aware of the victim and throughout criminal proceedings and for an appropriate time after 

In [17]:
celex_ids = []

for obj in response.objects:
    celex_ids.append(obj.properties['celex'])

celex_ids

['32011L0093', '32012L0029', '32005F0214', '32019D0417', '32012L0029']

In [13]:
celex_ids = []

for obj in response.objects:
    celex_ids.append(obj.properties['celex'])

celex_ids

['32012L0029', '32005F0214', '32011L0093', '32019D0417', '32008F0947']

In [17]:
from weaviate.classes.query import Filter

In [10]:
vectorstore = WeaviateVectorStore(
    client = weaviate_client,
    index_name = "Euro_Laws_hybrid",
    text_key="text",
    embedding = embedding_model
)

In [1]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("trespassing sentencing penalties fines imprisonment statutory sanctions jurisdiction")

NameError: name 'vectorstore' is not defined

In [21]:
docs[0].metadata.keys()

dict_keys(['total_chunks', 'document_length', 'eurovoc', 'chunk_number', 'subject_matter', 'celex', 'additional_info', 'authors', 'act_type', 'legal_basis', 'status', 'treaty', 'cites', 'act_name'])

In [9]:
def get_full_doc_weaviate(celex_id):

    response = eur_docs.query.fetch_objects(
    filters=Filter.by_property("celex").equal(celex_id),
    limit=1
)
    r = response.objects[0].properties

    # r is a dict with key, values for each doc
    return r    

In [10]:
def search_docs(query):
    celex_ids = []
    full_doc_info = """ """

    weaviate_client.connect()

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list(set(celex_ids))    

    for i , celex_id in enumerate(celex_ids):
        f_doc_meta = get_full_doc_weaviate(celex_id)

        full_doc_info += f"""

        doc {i} :

        'celex': {f_doc_meta['celex']}
        'status': {f_doc_meta['status']}
        'act_type': {f_doc_meta['act_type']}
        'treaty': {f_doc_meta['treaty']}

        full_doc :

        {f_doc_meta['full_doc']}
{"=="*15} "END OF DOC" {"=="*15}
        """
    weaviate_client.close()
    
    return full_doc_info    

In [8]:
# To use the hybrid search form the collections
eur_docs_hybrid = weaviate_client.collections.get("Euro_Laws_hybrid")

In [9]:
def get_embedding(query):
    embedding_model = SentenceTransformersEmbeddings("sentence-transformers/all-mpnet-base-v2")
    embed_query = embedding_model.embed_query(query)

    return embed_query

# hybrid search

In [10]:
def search_docs_hybrid(query):
    full_chunks_info = """ """

    # for debugging purpose
    weaviate_client.connect()

    # embed query
    embed_query = get_embedding(query)

    # Search docs using vectorstore
    response = eur_docs_hybrid.query.hybrid(
        query= query,
        vector= embed_query,
        alpha=0.5,
        limit=5
    )

    for i , obj in enumerate(response.objects):

        full_chunks_info += f"""

        chunk {i} :

        'celex': {obj.properties['celex']}
        'status': {obj.properties['status']}
        'act_type': {obj.properties['act_type']}
        'treaty': {obj.properties['treaty']}

        full_chunk :

        {obj.properties['text']}
{"=="*15} "END OF DOC" {"=="*15}
        """
        # for debugging purpose
        if len(full_chunks_info) > 5:
            print("docs found and retrieved successfully")
    print(len(full_chunks_info))

    weaviate_client.close()      

    return full_chunks_info  

In [11]:
display(Markdown(search_docs_hybrid("trespassing sentencing penalties fines imprisonment statutory sanctions jurisdiction")))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 630.37it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


docs found and retrieved successfully
docs found and retrieved successfully
docs found and retrieved successfully
docs found and retrieved successfully
docs found and retrieved successfully
16011


 

        chunk 0 :

        'celex': 32018R1100
        'status': In Force
        'act_type': Delegated Regulation
        'treaty': TFEU

        full_chunk :

        and medical products supply, and depending on the level of due diligence applied. Possible damages to EU interests: Measures to limit imports into USA or procurement to USA, prohibition of designation as primary dealer or as repository of US Government funds, denial of access to loans from US financial institutions or transfers through such institutions, prohibition of transactions in foreign exchange subject to the jurisdiction of the USA, export restrictions by USA, prohibition of property transactions subject to the jurisdiction of the USA, or refusal of assistance by EXIM-Bank, prohibitions and limitations to the opening and maintenance of correspondent accounts in the USA REGULATIONS Iranian Transactions and Sanctions Regulations Required compliance: Not to reexport any goods, technology, or services that (a) have been exported from the USA and (b) are subject to export control rules in the USA, if the export is made knowing or having reason to know that it is specifically intended for Iran or its Government. Goods substantially transformed into a foreign-made product outside the USA, and goods incorporated into such a product and representing less than 10 % of its value are not subject to the prohibition. Possible damages to EU interests: Imposition of civil penalties, fines and imprisonment.  ºC1 1. 31 CFR   (Code of Federal Regulations) Ch. V (7-1-95 edition) Part 515  Cuban Assets Control Regulations, subpart B (Prohibitions), E (Licenses, Authorizations and Statements of Licensing Policy) and G (Penalties) Required compliance: The prohibitions are consolidated in Title I of the Cuban Liberty and Democratic Solidarity Act of 1996, see above. Furthermore, requires the obtaining of licences and/or authorizations in respect of economic activities concerning Cuba. Possible damages to EU interests: Fines, forfeiture, imprisonment in cases of violation.
============================== "END OF DOC" ==============================
        

        chunk 1 :

        'celex': 32009D0316
        'status': In Force
        'act_type': Decision
        'treaty': TEU (1992)

        full_chunk :

        ANNEX B Common table of penalties and measures categories referred to in Article 4 Code Categories and sub-categories of offences 1000 open category Deprivation of freedom 1001 Imprisonment 1002 Life imprisonment 2000 open category Restriction of personal freedom 2001 Prohibition from frequenting some places 2002 Restriction to travel abroad 2003 Prohibition to stay in some places 2004 Prohibition from entry to a mass event 2005 Prohibition to enter in contact with certain persons through whatever means 2006 Placement under electronic surveillance (1) 2007 Obligation to report at specified times to a specific authority 2008 Obligation to stay/reside in a certain place 2009 Obligation to be at the place of residence on the set time 2010 Obligation to comply with the probation measures ordered by the court, including the obligation to remain under supervision 3000 open category Prohibition of a specific right or capacity 3001 Disqualification from function 3002 Loss/suspension of capacity to hold or to be appointed to public office 3003 Loss/suspension of the right to vote or to be elected 3004 Incapacity to contract with public administration 3005 Ineligibility to obtain public subsidies 3006 Cancellation of the driving licence (2) 3007 Suspension of driving licence 3008 Prohibition to drive certain vehicles 3009 Loss/suspension of the parental authority 3010 Loss/suspension of right to be an expert in court proceedings/witness under oath/juror 3011 Loss/suspension of right to be a legal guardian (3) 3012 Loss/suspension of right of decoration or title 3013 Prohibition to exercise professional, commercial or social activity 3014 Prohibition from working or activity with minors 3015 Obligation to close an establishment 3016 Prohibition to hold or to carry weapons 3017 Withdrawal of a hunting/fishing license 3018 Prohibition to issue cheques or to use payment/credit cards 3019 Prohibition to keep animals 3020 Prohibition to possess or use certain items other than weapons 3021 Prohibition to play certain games/sports 4000 open category Prohibition or expulsion from territory 4001 Prohibition from national territory 4002 Expulsion from national territory 5000 open category Personal obligation 5001 Submission to medical treatment or other forms of therapy 5002 Submission to a social-educational programme 5003 Obligation to be under the care/control of the family 5004 Educational measures 5005 Socio-judicial probation 5006 Obligation of training/working 5007 Obligation to provide judicial authorities with specific information 5008 Obligation to publish the judgment 5009 Obligation to compensate for the prejudice caused by the offence 6000 open category Penalty on personal property 6001 Confiscation 6002 Demolition 6003 Restoration 7000 open category Placing in an institution 7001 Placing in a psychiatric institution 7002 Placing in a detoxification institution 7003 Placing in an educational institution 8000 open category Financial penalty 8001 Fine 8002 Day-fine (4) 8003 Fine for the benefit of a special recipient (5) 9000 open category Working penalty 9001 Community service or work 9002 Community service or work accompanied with other restrictive measures 10000 open category Military penalty 10001 Loss of military rank (6) 10002 Expulsion from professional military service 10003 Military imprisonment 11000 open category Exemption/deferment of sentence/penalty, warning 12000 open
============================== "END OF DOC" ==============================
        

        chunk 2 :

        'celex': 32017L1371
        'status': In Force
        'act_type': Directive
        'treaty': TFEU

        full_chunk :

        damage or advantage as a basis for a maximum penalty, the Member State should ensure that the amount of damage or advantage is taken into account by its courts in the determination of sanctions for fraud and other criminal offences affecting the Union's financial interests. This Directive does not prevent Member States from providing for other elements which would indicate the serious nature of a criminal offence, for instance when the damage or advantage is potential, but of very considerable nature. However, for offences against the common VAT system, the threshold as of which the damage or advantage should be presumed to be considerable is, in conformity with this Directive, EUR 10 000 000. The introduction of minimum levels of maximum imprisonment sanctions is necessary in order to ensure equivalent protection of the Union's financial interests throughout the Union. The sanctions are intended to serve as a strong deterrent for potential offenders, with effect throughout the Union. (19) Member States should ensure that the fact that a criminal offence is committed within a criminal organisation as defined in Council Framework Decision 2008/841/JHA (10) is considered to be an aggravating circumstance in accordance with the applicable rules established by their legal systems. They should ensure that the aggravating circumstance is made available to judges for their consideration when sentencing offenders, although there is no obligation on judges to take the aggravating circumstance into account in their sentence. Member States are not obliged to provide for the aggravating circumstance where national law provides for the criminal offences as defined in Framework Decision 2008/841/JHA to be punishable as a separate criminal offence and this may lead to more severe sanctions. (20) Given, in particular, the mobility of perpetrators and of the proceeds stemming from illegal activities at the expense of the Union's financial interests, as well as the complex cross-border investigations which this entails, each Member State should establish its jurisdiction in order to enable it to counter such activities. Each Member State should thereby ensure that its jurisdiction covers criminal offences which are committed using information and communication technology accessed from its territory. (21) Given the possibility of multiple jurisdictions for cross-border criminal offences falling under the scope of this Directive, the Member States should ensure that the principle of ne bis in idem is respected in full in the application of national law transposing this Directive. (22) Member States should lay down rules concerning limitation periods necessary in order to enable them to counter illegal activities at the expense of the Union's financial interests. In cases of criminal offences punishable by a maximum sanction of at least four years of imprisonment, the limitation period should be at least five years from the time when the criminal offence was committed. This should be without prejudice to those Member States which do not set limitation periods for investigation, prosecution and enforcement. (23) Without prejudice to the rules on cross-border cooperation and mutual legal assistance in criminal matters and to other rules under Union
============================== "END OF DOC" ==============================
        

        chunk 3 :

        'celex': 32017L1371
        'status': In Force
        'act_type': Directive
        'treaty': TFEU

        full_chunk :

        offences that are not of a serious nature, in cases where intent is presumed under national law. (13) Some criminal offences against the Union's financial interests are in practice often closely related to the criminal offences covered by Article 83(1) of the Treaty on the Functioning of the European Union (TFEU) and Union legislative acts that are based on that provision. Coherence between such legislative acts and this Directive should therefore be ensured in the wording of this Directive. (14) Insofar as the Union's financial interests can be damaged or threatened by conduct attributable to legal persons, legal persons should be liable for the criminal offences, as defined in this Directive, which are committed on their behalf. (15) In order to ensure equivalent protection of the Union's financial interests throughout the Union by means of measures which should act as a deterrent, Member States should provide for certain types and levels of sanctions when the criminal offences defined in this Directive are committed. The levels of sanctions should not go beyond what is proportionate for the offences. (16) As this Directive provides for minimum rules, Member States are free to adopt or maintain more stringent rules for criminal offences affecting the Union's financial interests. (17) This Directive does not affect the proper and effective application of disciplinary measures or penalties other than of a criminal nature. Sanctions that cannot be equated to criminal sanctions, which are imposed on the same person for the same conduct, can be taken into account when sentencing that person for a criminal offence defined in this Directive. For other sanctions, the principle of prohibition of being tried or punished twice in criminal proceedings for the same criminal offence (ne bis in idem) should be fully respected. This Directive does not criminalise behaviour which is not also subject to disciplinary penalties or other measures concerning a breach of official duties, in cases where such disciplinary penalties or other measures can be applied to the persons concerned. (18) Sanctions with regard to natural persons should, in certain cases, provide for a maximum penalty of at least four years of imprisonment. Such cases should include at least those involving considerable damage done or advantage gained whereby the damage or advantage should be presumed to be considerable when it involves more than EUR 100 000. Where a Member State's law does not provide for an explicit threshold for considerable damage or advantage as a basis for a maximum penalty, the Member State should ensure that the amount of damage or advantage is taken into account by its courts in the determination of sanctions for fraud and other criminal offences affecting the Union's financial interests. This Directive does not prevent Member States from providing for other elements which would indicate the serious nature of a criminal offence, for instance when the damage or advantage is potential, but of very considerable nature. However, for offences against the common VAT system, the threshold as of which the damage or advantage should be
============================== "END OF DOC" ==============================
        

        chunk 4 :

        'celex': 32014L0057
        'status': In Force
        'act_type': Directive
        'treaty': TFEU (2008)

        full_chunk :

        mutandis. Article 7 Criminal penalties for natural persons 1. Member States shall take the necessary measures to ensure that the offences referred to in Articles 3 to 6 are punishable by effective, proportionate and dissuasive criminal penalties. 2. Member States shall take the necessary measures to ensure that the offences referred to in Articles 3 and 5 are punishable by a maximum term of imprisonment of at least four years. 3. Member States shall take the necessary measures to ensure that the offence referred to in Article 4 is punishable by a maximum term of imprisonment of at least two years. Article 8 Liability of legal persons 1. Member States shall take the necessary measures to ensure that legal persons can be held liable for offences referred to in Articles 3 to 6 committed for their benefit by any person, acting either individually or as part of an organ of the legal person, and having a leading position within the legal person based on: (a) a power of representation of the legal person; (b) an authority to take decisions on behalf of the legal person; or (c) an authority to exercise control within the legal person. 2. Member States shall also take the necessary measures to ensure that legal persons can be held liable where the lack of supervision or control, by a person referred to in paragraph 1, has made possible the commission of an offence referred to in Articles 3 to 6 for the benefit of the legal person by a person under its authority. 3. Liability of legal persons under paragraphs 1 and 2 shall not exclude criminal proceedings against natural persons who are involved as perpetrators, inciters or accessories in the offences referred to in Articles 3 to 6. Article 9 Sanctions for legal persons Member States shall take the necessary measures to ensure that a legal person held liable pursuant to Article 8 is subject to effective, proportionate and dissuasive sanctions, which shall include criminal or non-criminal fines and may include other sanctions, such as: (a) exclusion from entitlement to public benefits or aid; (b) temporary or permanent disqualification from the practice of commercial activities; (c) placing under judicial supervision; (d) judicial winding-up; (e) temporary or permanent closure of establishments which have been used for committing the offence. Article 10 Jurisdiction 1. Member States shall take the necessary measures to establish their jurisdiction over the offences referred to in Articles 3 to 6 where the offence has been committed: (a) in whole or in part within their territory; or (b) by one of their nationals, at least in cases where the act is an offence where it was committed. 2. A Member State shall inform the Commission where it decides to establish further jurisdiction over the offences referred to in Articles 3 to 6 committed outside its territory where: (a) the offender is an habitual resident in its territory; or (b) the offence is committed for the benefit of a legal person established in its
============================== "END OF DOC" ==============================
        

In [13]:
response = eur_docs_hybrid.query.hybrid(
    query="rape sentencing guidelines, punishment for rape, statutory penalties for rape, judicial discretion in rape cases",
    vector= quer_embed ,
    alpha=0.5,
    limit=5
)

In [19]:
response.objects[0].properties

{'authors': 'European Commission',
 'treaty': 'TEC (1992)',
 'text': "Avis juridique important|31995D023295/232/EC: Commission Decision of 27 June 1995 on the organization of a temporary experiment under Council Directive 69/208/EEC in order to establish conditions to be satisfied by the seed of hybrids and varietal associations of swede rape and turnip rape Official Journal L 154 , 05/07/1995 P. 0022 - 0025COMMISSION DECISION of 27 June 1995 on the organization of a temporary experiment under Council Directive 69/208/EEC in order to establish conditions to be satisfied by the seed of hybrids and varietal associations of swede rape and turnip rape (95/232/EC)THE COMMISSION OF THE EUROPEAN COMMUNITIES, Having regard to the Treaty establishing the European Community, Having regard to Council Directive 69/208/EEC of 30 June 1969 on the marketing of seed of oil and fibre plants (1), as last amended by the Act of Accession of Austria, Finland and Sweden, and in particular Article 12a thereo

In [14]:
for obj in response.objects:
    print (obj.properties)

{'authors': 'European Commission', 'treaty': 'TEC (1992)', 'text': "Avis juridique important|31995D023295/232/EC: Commission Decision of 27 June 1995 on the organization of a temporary experiment under Council Directive 69/208/EEC in order to establish conditions to be satisfied by the seed of hybrids and varietal associations of swede rape and turnip rape Official Journal L 154 , 05/07/1995 P. 0022 - 0025COMMISSION DECISION of 27 June 1995 on the organization of a temporary experiment under Council Directive 69/208/EEC in order to establish conditions to be satisfied by the seed of hybrids and varietal associations of swede rape and turnip rape (95/232/EC)THE COMMISSION OF THE EUROPEAN COMMUNITIES, Having regard to the Treaty establishing the European Community, Having regard to Council Directive 69/208/EEC of 30 June 1969 on the marketing of seed of oil and fibre plants (1), as last amended by the Act of Accession of Austria, Finland and Sweden, and in particular Article 12a thereof,

In [15]:
retriever = vectorstore.as_retriever(
    search_type="hybrid",
    search_kwargs={
        "k": 5,
        "alpha": 0.5
    }
)
docs = retriever.invoke("trespassing sentencing penalties fines imprisonment statutory sanctions jurisdiction")

ValidationError: 1 validation error for VectorStoreRetriever
  Value error, search_type of hybrid not allowed. Valid values are: ('similarity', 'similarity_score_threshold', 'mmr') [type=value_error, input_value={'vectorstore': <langchai... {'k': 5, 'alpha': 0.5}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [26]:
weaviate_client.close()

In [ ]:
display(Markdown(search_docs("trespassing sentencing penalties fines imprisonment statutory sanctions jurisdiction")))

In [12]:
display(Markdown(search_docs("Drug dealing Sentences")))

 

        doc 0 :

        'celex': 31997R2046
        'status': Not in Force
        'act_type': Regulation
        'treaty': TEC (1992)

        full_doc :

        Avis juridique important|31997R2046Council Regulation (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addiction Official Journal L 287 , 21/10/1997 P. 0001 - 0005COUNCIL REGULATION (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addictionTHE COUNCIL OF THE EUROPEAN UNION,Having regard to the Treaty establishing the European Community, and in particular Article 130w thereof,Having regard to the proposal from the Commission (1),Acting in accordance with the procedure laid down in Article 189c of the Treaty (2),Whereas the impact on the structures of a developing society of an economy based on the production of drugs, or which derives a substantial revenue from them, undermines a country's smooth integration into the world economy;Whereas the breakdown of social structures in developing countries due to drug consumption and the related industry is detrimental to sustainable social development and the attainment of the goals of Community policy in the sphere of development cooperation as defined in Article 130u of the Treaty;Whereas as part of the effort to combat the supply of drugs it is essential, in particular, to effect a radical reduction of poverty in the south and to offer the population a lawful alternative to the growing of illegal crops;Whereas institutional support should be given to those developing countries which so request so that they can combat drugs more effectively;Whereas in a communication to the Parliament and to the Council dated 23 June 1994, the Commission presented its guidelines for a European Union plan of action on drugs for 1995-1999, including measures at international level;Whereas Parliament stated its views on the guidelines in its Opinion on the communication, adopted on 15 June 1995;Whereas the Fourth ACP-EC Convention and the cooperation, association and partnership agreements concluded by the European Community with developing countries contain clauses on cooperation to curb drug abuse and drug trafficking, the monitoring of trade in precursors, chemical products and psychotropic substances and the exchange of relevant information, including measures in the field of money laundering; whereas there is a relationship between the campaign against drugs and drug addiction and the aims of the cooperation policy pursued by the Community and its developing-country partners;Whereas the international community's strategy to curb drug abuse and drug trafficking is based on universal accession to the Single Convention on narcotic drugs of 1961, as amended by the Protocol of 1972, the Convention on psychotropic substances of 1971 and the International Convention against illicit traffic in narcotic drugs and psychotropic substances of 1988, and on the systematic implementation of those conventions at national and international level;Whereas the European Community is a party to the Convention of 1988, in particular by virtue of Article 12 of that Convention, and has adopted Community legislation based on the recommendations of the Chemicals Action Task Force set up by the G7 and the President of the Commission in 1989, the effectiveness of which would be generally enhanced by the adoption of the relevant legislation and procedures in other parts of the world;Whereas effective action against drugs must also encompass measures against the laundering of money from drug trafficking, such as the adoption of a suitable legal framework and appropriate mechanisms in the countries concerned;Whereas human rights must be duly respected in implementing measures under this Regulation;Whereas the Member States of the European Community have endorsed the policy statement and general plan of action adopted by the UN General Assembly at its 17th special session;Whereas a financial reference amount within the meaning of point 2 of the Declaration by the European Parliament, the Council and the Commission of 6 March 1995 is included in this Regulation for the period 1998-2000, without thereby affecting the powers of the budgetary authority as they are defined by the Treaty,HAS ADOPTED THIS REGULATION:Article 1 In the framework of its development cooperation policy and taking account of the harmful effects on development efforts of the production, trading and consumption of drugs, the Community shall carry out cooperation activities in the field of drugs and drug addiction in developing countries, giving priority to those which have demonstrated political will at the highest level to solve their drug problem. The existence of such will may be demonstrated inter alia by ratification of the Single Convention of 1961 as amended by the Protocol of 1972, the Convention of 1971 and the Convention of 1988. Commitment on the part of developing countries shall take the form, inter alia, of the implementation of domestic legislation against laundering of money generated through illicit drugs.Article 2 The assistance provided under this Regulation shall complement and reinforce assistance provided under other instruments of development cooperation.Article 3 The Community shall give priority at the request of a partner country to supporting the preparation of a national drug control master plan, in close consultation with the United Nations International Drug Control Programme (UNDCP). These plans will identify objectives, strategies and priorities in the campaign against drugs and the related resource requirements (including financial requirements), thus establishing an integrated, multidisciplinary and multisectoral approach designed to maximize the efficiency of national drug control programmes and international assistance.The prevention of drug addiction, together with demand reduction, shall be addressed in a consistent policy comprising education and objective information about the consequences of addiction, targeted especially at young people.Community cooperation shall take place in a spirit of dialogue reflecting the genuine cultural differences which affect the perception of drug-related problems, this being crucial to ensure the social and political viability of drug control strategies.Article 4 Preferably operating within the strategic framework established by the national plans, the Community shall also support specific operations capable of a measurable impact (i.e. effective and tangible results within a time-limit set in advance) in the following areas:- development of institutional capacity, in particular for the implementation of:- National Drug Control Plans by developing countries,- agreements between the Community and certain developing countries, in particular to combat the diversion of chemical precursors and to curb money laundering,- demand reduction, in particular through analysis of local patterns, the introduction of measures to control trade in and consumption of narcotics and psychotropic substances, treatment and reintegration of drug addicts, as well as risk limitation. These measures must be integrated into policies on health and education, development and combating poverty and social and economic exclusion,- promotion of pilot alternative development projects, conceived as a process by which the production of illicit drug crops is eventually both combatted and eliminated through appropriate rural development measures in the context of sustained national economic growth. These projects shall comprise social and economic measures which take into account factors contributing to illicit production as well as measures which may facilitate improved use of commercial preferences. In this connection, it shall be systematically examined whether it is possible to make more use of other Community financial instruments (e.g. ALA) and the European Development Fund for alternative development projects,- financing of studies, seminars and fora for the exchange of experience in the above fields.Particular attention shall be paid to the participation of local people and target groups in identifying, planning and carrying out operations.The Community shall only finance projects in which respect for human rights is guaranteed.Article 5 The cooperation partners eligible for financial support under this Regulation shall be regional and international organizations, in particular UNDCP, local- and Member State-based non-governmental organizations, national, provincial and local government departments and agencies, community-based organizations, institutes and public and private operators.Article 6 1. The instruments to be employed in the course of the operations referred to in Articles 3 and 4 shall include studies, technical assistance, training or other services, supplies and works, along with audits and evaluation and monitoring missions.2. According to the needs of the operations concerned, Community financing may cover both capital investment, other than the purchase of real estate, and operating costs in foreign or local currency. However, with the exception of training programmes, operating costs may normally be covered only during the start-up phase and on a degressive basis.3. A financial contribution from the partners defined in Article 5 shall be sought for each cooperation operation. Their contribution will be requested within the limits of the possibilities available to the parties concerned and depending on the nature of the operation concerned.4. A financial contribution from the local partners, particularly in respect of operating costs, shall be sought as a matter of priority in the case of projects intended to launch long term activities, so as to ensure the viability of such projects once Community funding comes to an end.5. Opportunities may be sought for cofinancing with other fund providers, and especially with Member States.6. The Commission will ensure that the Community character of the aid provided under this Regulation is highlighted.7. In order to achieve the objectives of consistency and complementarity referred to in the Treaty and with the aim of guaranteeing optimum effectiveness of all these operations, the Commission may take all necessary coordination measures, including in particular:(a) a system for the systematic exchange and analysis of information on operations financed and those which the Community and the Member States propose to finance;(b) on-the-spot coordination of the implementation of operations through regular meetings and exchange of information between representatives of the Commission and of the Member States in the beneficiary country.8. In order to obtain the greatest possible impact globally and nationally, the Commission, in liaison with the Member States, shall take any initiative necessary for ensuring proper coordination and close collaboration with the beneficiary countries and the providers of funds and other international agencies involved, in particular those forming part of the United Nations system and more specifically the UNDCP.Article 7 Financial support under this Regulation shall take the form of grants.Article 8 The financial reference amount for the implementation of this programme during the period 1998-2000 shall be ECU 30 million.Annual appropriations shall be authorized by the budgetary authority within the limits of the financial perspectives.Article 9 1. The Commission shall be responsible for appraising, approving and managing operations covered by this Regulation in accordance with the budgetary and other procedures in force, and in particular those laid down in the Financial Regulation applicable to the general budget of the European Communities.2. Projects and programme appraisal shall take into account the following factors:- effectiveness and viability of operations,- cultural, social, gender and environmental aspects,- institutional development necessary to achieve project goals,- experience gained from operations of the same kind.3. Decisions relating to grants of more than ECU 2 million for individual operations financed under this Regulation and any changes resulting in an increase of more than 20 % in the sum initially approved for such an operation shall be adopted under the procedure laid down in Article 10.The Commission shall inform the Committee referred to in Article 10 succinctly of the financing decisions which it intends to take with regard to projects and programmes of less than ECU 2 million in value. The information shall be made available not later than one week before the decision is taken.4. The Commission shall be authorized to approve, without recourse to the opinion of the Committee provided for in Article 10, any supplementary commitments needed for covering expected or real cost overruns in connection with the operations, where the overrun or additional requirement is less than or equal to 20 % of the initial commitment fixed by the financing decision.5. All financing agreements or contracts concluded under this Regulation shall provide for the Commission and the Court of Auditors to conduct on-the-spot checks according to the usual procedures laid down by the Commission under the rules in force, and in particular those of the Financial Regulation applicable to the general budget of the European Communities.6. Where operations are the subject of financing agreements between the Community and the recipient country, such agreement shall stipulate that the payment of taxes, duties or any other charges is not to be covered by the Community.7. Participation in invitations to tender and the award of contracts shall be open on equal terms to natural and legal persons of the Member States and of the recipient country. It may be extended to other developing countries.8. Supplies shall originate in the Member States, the recipient country or other developing countries. In exceptional cases, where circumstances warrant, supplies may originate elsewhere.9. Particular attention will be given to:- the pursuit of cost-effectiveness and sustainable impact in project design,- the clear definition and monitoring of objectives and indications of achievement for all projects.Article 10 1. The Commission shall be assisted by the geographically-determined committee competent for development.2. The representative of the Commission shall submit to the committee a draft of the measures to be taken. The committee shall deliver its opinion on the draft, within a time limit which the chairman may lay down according to the urgency of the matter. The opinion shall be delivered by the majority laid down in Article 148 (2) of the Treaty in the case of decisions which the Council is required to adopt on a proposal from the Commission. The votes of the representatives of the Member States within the committee shall be weighted in the manner set out in that Article. The Chairman shall not vote.The Commission shall adopt the measures envisaged if they are in accordance with the opinion of the committee.If the measures envisaged are not in accordance with the opinion of the committee, or if no opinion is delivered, the Commission shall without delay submit to the Council a proposal relating to the measures to be taken. The Council shall act by a qualified majority.If, on the expiry of a period of three months from the date of referral to the Council, the Council has not acted, the proposed measures shall be adopted by the Commission.3. An exchange of views shall take place once a year on the basis of a presentation by the representative of the Commission of the general guidelines for the operations to be carried out in the year ahead, in the framework of a joint meeting of the committees referred to in paragraph 1.Article 11 1. At the end of each budget year, the Commission shall present a report to Parliament and the Council summarizing the operations financed in the course of that year and evaluating the implementation of this Regulation over that period.The summary shall in particular contain information about those with whom contracts have been concluded.2. The Commission shall regularly assess operations financed by the Community with a view to establishing whether the objectives aimed at by such operations have been achieved and to providing guidelines for improving the effectiveness of future operations. The Commission shall submit to the Committee referred to in Article 10 a summary of the assessments made which, if appropriate, may be examined by the Committee. The assessment reports shall be made available to any Member States requesting them.3. The Commission shall inform the Member States, at the latest one month after its decision, of the operations and projects approved, stating their cost and nature, the recipient country and partners.Article 12 1. This Regulation shall enter into force on the third day following that of its publication in the Official Journal of the European Communities.2. Three years after this Regulation enters into force, the Commission shall submit to the European Parliament and the Council an overall assessment of operations financed by the Community under this Regulation together with suggestions regarding the future of this Regulation and, where necessary, proposals for amending or terminating it.This Regulation shall be binding in its entirety and directly applicable in all Member States.Done at Luxembourg, 13 October 1997.For the CouncilThe PresidentJ.-C. JUNCKER(1) OJ C 242, 19. 9. 1995, p. 8.(2) Opinion of the European Parliament of 19 April 1996 (OJ C 141, 13. 5. 1996, p. 252), Council Common Position of 22 November 1996, (OJ C 6, 9. 1. 1997, p. 1) and Decision of the European Parliament of 13 March 1997 (OJ C 115/97, 14. 4. 1997, p. 127).
============================== "END OF DOC" ==============================
        

        doc 1 :

        'celex': 32014D0688
        'status': Not in Force
        'act_type': Decision_IMPL
        'treaty': TFEU (2008)

        full_doc :

        1.10.2014 EN Official Journal of the European Union L 287/22 COUNCIL IMPLEMENTING DECISION of 25 September 2014 on subjecting 4-iodo-2,5-dimethoxy-N-(2-methoxybenzyl)phenethylamine (25I-NBOMe), 3,4-dichloro-N-[[1-(dimethylamino)cyclohexyl]methyl]benzamide (AH-7921), 3,4-methylenedioxypyrovalerone (MDPV) and 2-(3-methoxyphenyl)-2-(ethylamino)cyclohexanone (methoxetamine) to control measures (2014/688/EU) THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, Having regard to Council Decision 2005/387/JHA of 10 May 2005 on the information exchange, risk-assessment and control of new psychoactive substances (1), and in particular Article 8(3) thereof, Having regard to the proposal from the European Commission, Whereas: (1) Risk assessment reports on the new psychoactive substances 4-iodo-2,5-dimethoxy-N-(2-methoxybenzyl)phenethylamine (25I-NBOMe), 3,4-dichloro-N-[[1-(dimethylamino)cyclohexyl]methyl]benzamide (AH-7921), 3,4-methylenedioxypyrovalerone (MDPV) and 2-(3-methoxyphenyl)-2-(ethylamino)cyclohexanone (methoxetamine) were drawn up in compliance with Decision 2005/387/JHA by a special session of the extended Scientific Committee of the European Monitoring Centre for Drugs and Drug Addiction (EMCDDA), and were subsequently submitted to the Commission and to the Council on 23 April 2014. (2) 25I-NBOMe, AH-7921, MDPV and methoxetamine had not been under assessment at the United Nations' level by the time the risk assessment was requested at Union level, but they were evaluated in June 2014 by the Expert Committee on Drug Dependence of the World Health Organization. (3) 25I-NBOMe, AH-7921, MDPV and methoxetamine have no established or acknowledged medical use (human or veterinary). Apart from their use in analytical reference materials, and in scientific research investigating their chemistry, pharmacology and toxicology as a result of their emergence on the drug market  and, in the case of 25I-NBOMe, also in the field of neurochemistry  there is no indication that they are being used for other purposes. (4) 25I-NBOMe is a potent synthetic derivative of 2,5-dimethoxy-4-iodophenethylamine (2C-I), a classical serotonergic hallucinogen, which was subject to risk assessment and to control measures and criminal sanctions at Union level from 2003 by Council Decision 2003/847/JHA (2). (5) The specific physical effects of 25I-NBOMe are difficult to determine because there are no published studies assessing its acute and chronic toxicity, its psychological and behavioural effects, and dependence potential, and because of the limited information and data available. Clinical observations of individuals who have used this substance suggest that it has hallucinogenic effects and has the potential for inducing severe agitation, confusion, intense auditory and visual hallucinations, aggression, violent accidents and self-induced trauma. (6) There have been four deaths associated with 25I-NBOMe registered in three Member States. Severe toxicity associated with its use has been reported in four Member States, which notified 32 non-fatal intoxications. If this new psychoactive substance were to become more widely available and used, the implications for individual and public health could be significant. There is no information available on the social risks associated with 25I-NBOMe. (7) 22 Member States and Norway have reported to the EMCDDA and European Police Office (Europol) that they detected 25I-NBOMe. No prevalence data is available on the use of 25I-NBOMe, but the limited information that exists suggests that it may be consumed in a wide range of settings, such as at home, in bars, nightclubs and at music festivals. (8) 25I-NBOMe is openly marketed and sold on the internet as a research chemical and information from seizures, collected samples, user websites and internet retailers suggests that it is being sold as a drug in its own right and also marketed as a legal replacement for LSD. EMCDDA identified more than 15 internet retailers selling this substance, who may be based within the Union and China. (9) The risk assessment report reveals that there is limited scientific evidence available on 25I-NBOMe and points out that further research would be needed to determine the health and social risks that it poses. However, the available evidence and information provides sufficient ground for subjecting 25I-NBOMe to control measures across the Union. As a result of the health risks that it poses, as documented by its detection in several reported fatalities, of the fact that users may unknowingly consume it and of the lack of medical value or use of the substance, 25I-NBOMe should be subjected to control measures across the Union. (10) Since six Member States control 25I-NBOMe under national legislation complying with the obligations of the 1971 United Nations Convention on Psychotropic Substances, and seven Member States use other legislative measures to control it, subjecting this substance to control measures across the Union would help avoid the emergence of obstacles to cross-border law enforcement and judicial cooperation, and would help protect against the risks that its availability and use can pose. (11) AH-7921 is a structurally atypical synthetic opioid analgesic commonly known by internet suppliers, user websites and media as doxylam. It can be easily confused with doxylamine, an antihistaminic medicine with sedative-hypnotic properties, which could lead to unintentional overdoses. (12) The specific physical effects of AH-7921 are difficult to determine because there are no published studies assessing its acute and chronic toxicity, its psychological, behavioural effects, and dependence potential, as well as the limited information and data available. Based on user reports, the effects of AH-7921 appear to resemble those of classical opioids with the feeling of mild euphoria, itchiness and relaxation; nausea appears to be a typical adverse effect. In addition to self-experimentation with AH-7921, as well as recreational use, some of the users report self-medicating with this new drug to relieve pain, others to alleviate withdrawal symptoms due to cessation of the use of other opioids. This may indicate a potential of AH-7921 to spread among the injecting opioid population. (13) There is no prevalence data on the use of AH-7921, but the information available suggests that it is not widely used, and that when it is used, that use is in the home environment. (14) 15 fatalities were recorded in three Member States between December 2012 and September 2013 where AH-7921, alone or in combination with other substances, was detected in post-mortem samples. While it is not possible to determine with certainty the role of AH-7921 in all of those fatalities, in some cases it has been specifically noted in the cause of death. One Member State reported six non-fatal intoxications associated with AH-7921. If this new psychoactive substance were to become more widely available and used, the implications for individual and public health could be significant. There is no information available on the social risks associated with AH-7921. (15) The risk assessment report reveals that there is limited scientific evidence available on AH-7921 and points out that further research would be needed to determine the health and social risks that it poses. However, the available evidence and information provides sufficient ground for subjecting AH-7921 to control measures across the Union. As a result of the health risks that it poses, as documented by its detection in several reported fatalities, of the fact that users may unknowingly consume it, and of the lack of medical value or use of the substance, AH-7921 should be subjected to control measures across the Union. (16) Since one Member State controls AH-7921 under national legislation complying with the obligations of the 1971 United Nations Convention on Psychotropic Substances and five Member States use other legislative measures to control it, subjecting this substance to control measures across the Union would help avoid the emergence of obstacles in cross-border law enforcement and judicial cooperation, and would help protect against the risks that its availability and use can pose. (17) MDPV is a ring-substituted synthetic derivative of cathinone chemically related to pyrovalerone, which are both subject to control under the 1971 United Nations Convention on Psychotropic Substances. (18) Information on the chronic and acute toxicity associated with MDPV, as well as on psychological and behavioural effects, and on dependence potential, is not collected uniformly across the Union. Information from published studies, confirmed by clinical cases, suggests that the psychopharmacological profile observed for MDPV is similar to that for cocaine and methamphetamine, albeit more potent and longer lasting. Furthermore, MDPV was found to be 10 times more potent in its ability to induce locomotor activation, tachycardia and hypertension. (19) Users' websites indicate that its acute toxicity can provoke adverse effects on humans, similar to those associated with other stimulants. These include paranoid psychosis, tachycardia, hypertension, diaphoresis, breathing problems, severe agitation, auditory and visual hallucinations, profound anxiety, hyperthermia, violent outbursts and multiple organ dysfunctions. (20) 108 fatalities were registered in eight Member States and Norway between September 2009 and August 2013, where MDPV has been detected in post-mortem biological samples or implicated in the cause of death. A total of 525 non-fatal intoxications associated with MDPV have been reported by eight Member States. If this new psychoactive substance were to become more widely available and used, the implications for individual and public health could be significant. (21) The detection of MDPV has also been reported in biological samples related to fatal and non-fatal road traffic accidents, or driving under the influence of drugs, in four Member States since 2009. (22) MDPV has been present in the Union drug market since November 2008 and 27 Member States, Norway and Turkey reported multi-kilogram seizures of the substance. MDPV is being sold as a substance in its own right, but it has also been detected in combination with other substances. It is widely available from internet suppliers and retailers, head shops and street-level dealers. There are some indications that suggest a degree of organisation in the tableting and distribution of this substance in the Union. (23) The risk assessment report reveals that further research would be needed to determine the health and social risks posed by MDPV. However, the available evidence and information provides sufficient ground for subjecting MDPV to control measures across the Union. As a result of the health risks that it poses, as documented by its detection in several reported fatalities, of the fact that users may unknowingly consume it, and of the lack of medical value or use of the substance, MDPV should be subjected to control measures across the Union. (24) Since 21 Member States control MDPV under national legislation complying with the obligations of the 1971 United Nations Convention on Psychotropic Substances and four Member States use other legislative measures to control it, subjecting this substance to control measures across the Union would help avoid the emergence of obstacles in cross-border law enforcement and judicial cooperation, and would protect against the risks that its availability and use can pose. (25) Methoxetamine is an arylcyclohexylamine substance which is chemically similar to ketamine and the internationally controlled substance phencyclidine (PCP). Like ketamine and PCP, it has dissociative properties. (26) There are no studies assessing the chronic and acute toxicity associated with methoxetamine, as well as its psychological and behavioural effects, and dependence potential. Self-reported experiences from user websites suggest adverse effects similar to ketamine intoxication. These include nausea and severe vomiting, difficulty in breathing, seizures, disorientation, anxiety, catatonia, aggression, hallucination, paranoia and psychosis. In addition, acute methoxetamine intoxications may include stimulant effects (agitation, tachycardia and hypertension) and cerebral features, which are not expectable with acute ketamine intoxication. (27) Twenty deaths associated with methoxetamine were reported by six Member States that detected the substance in post-mortem samples. Used alone or in combination with other substances, methoxetamine was detected in 20 non-fatal intoxications reported by five Member States. If this new psychoactive substance were to become more widely available and used, the implications for individual and public health could be significant. (28) 23 Member States, Turkey and Norway have reported that they detected methoxetamine, since November 2010. Information suggests that it is sold and used as a substance in its own right, but it is also sold as a legal replacement for ketamine by internet retailers, head shops and street-level drug dealers. (29) Multi-kilogram quantities in powder form were seized within the Union, but there is no information on the possible involvement of organised crime. The manufacture of methoxetamine does not require sophisticated equipment. (30) Prevalence data are limited to non-representative studies in two Member States. Those studies suggest that the prevalence of the use of methoxetamine is lower than that of ketamine. The available information suggests that it may be consumed in a wide range of settings, including at home, in bars, nightclubs and at music festivals. (31) The risk assessment report reveals that further research would be needed to determine the health and social risks posed by methoxetamine. However, the available evidence and information provides sufficient grounds for subjecting methoxetamine to control measures across the Union. As a result of the health risks that it poses, as documented by its detection in several reported fatalities, of the fact that users may unknowingly consume it, and of the lack of medical value or use, methoxetamine should be subjected to control measures across the Union. (32) Since nine Member States control methoxetamine under national legislation complying with the obligations of the 1971 United Nations Convention on Psychotropic Substances and nine Member States use other legislative measures to control it, subjecting this substance to control measures across the Union would help avoid the emergence of obstacles in cross-border law enforcement and judicial cooperation, and would protect against the risks that its availability and use can pose. (33) Decision 2005/387/JHA reserves to the Council implementing powers with a view to giving a quick and expertise-based response at the Union level to the emergence of new psychoactive substances detected and reported by the Member States, by submitting those substances to control measures across the Union. As the conditions and procedure for triggering the exercise of such implementing powers have been met, an implementing decision should be adopted in order to put 25I-NBOMe, AH-7921, MDPV and methoxetamine under control across the Union, HAS ADOPTED THIS DECISION: Article 1 The following new psychoactive substances shall be subjected to control measures across the Union: (a) 4-iodo-2,5-dimethoxy-N-(2-methoxybenzyl) phenethylamine(25I-NBOMe); (b) 3,4-dichloro-N-[[1-dimethylamino) cyclohexyl]methyl] benzamide (AH-7921); (c) 3,4-methylenedioxypyrovalerone (MDPV); (d) 2-(3-methoxyphenyl)-2-(ethylamino)cyclohexanone (methoxetamine). Article 2 By 2 October 2015, Member States shall subject in accordance with their national legislation, the new psychoactive substances referred to in Article 1 to control measures and criminal penalties, as provided for under their legislation complying with their obligations under the 1971 United Nations Convention on Psychotropic Substances. Article 3 This Decision shall enter into force on the twentieth day following that of its publication in the Official Journal of the European Union. Done at Brussels, 25 September 2014. For the Council The President F. GUIDI (1) OJ L 127, 20.5.2005, p. 32. (2) Council Decision 2003/847/JHA of 27 November 2003 concerning control measures and criminal sanctions in respect of the new synthetic drugs 2C-I, 2C-T-2, 2C-T-7 and TMA-2 (OJ L 321, 6.12.2003, p. 64).
============================== "END OF DOC" ==============================
        

        doc 2 :

        'celex': 32004F0757
        'status': In Force
        'act_type': Decision_FRAMW
        'treaty': TEU (1992)

        full_doc :

        11.11.2004 EN Official Journal of the European Union L 335/8 COUNCIL FRAMEWORK DECISION 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the field of illicit drug trafficking THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on European Union, and in particular Article 31(e) and Article 34(2)(b) thereof, Having regard to the proposal from the Commission (1), Having regard to the opinion of the European Parliament (2), Whereas: (1) Illicit drug trafficking poses a threat to health, safety and the quality of life of citizens of the European Union, and to the legal economy, stability and security of the Member States. (2) The need for legislative action to tackle illicit drug trafficking has been recognised in particular in the Action Plan of the Council and the Commission on how best to implement the provisions of the Amsterdam Treaty on an area of freedom, security and justice (3), adopted by the Justice and Home Affairs Council in Vienna on 3 December 1998, the conclusions of the Tampere European Council of 15 and 16 October 1999, in particular point 48 thereof, the European Union's Drugs Strategy (2000-2004) endorsed by the Helsinki European Council from 10 to 12 December 1999 and the European Union's Action Plan on Drugs (2000-2004) endorsed by the European Council in Santa Maria da Feira on 19 and 20 June 2000. (3) It is necessary to adopt minimum rules relating to the constituent elements of the offences of illicit trafficking in drugs and precursors which will allow a common approach at European Union level to the fight against such trafficking. (4) By virtue of the principle of subsidiarity, European Union action should focus on the most serious types of drug offence. The exclusion of certain types of behaviour as regards personal consumption from the scope of this Framework Decision does not constitute a Council guideline on how Member States should deal with these other cases in their national legislation. (5) Penalties provided for by the Member States should be effective, proportionate and dissuasive, and include custodial sentences. To determine the level of penalties, factual elements such as the quantities and the type of drugs trafficked, and whether the offence was committed within the framework of a criminal organisation, should be taken into account. (6) Member States should be allowed to make provision for reducing the penalties when the offender has supplied the competent authorities with valuable information. (7) It is necessary to take measures to enable the confiscation of the proceeds of the offences referred to in this Framework Decision. (8) Measures should be taken to ensure that legal persons can be held liable for the criminal offences referred to by this Framework Decision which are committed for their benefit. (9) The effectiveness of the efforts made to tackle illicit drug trafficking depends essentially on the harmonisation of the national measures implementing this Framework Decision, HAS DECIDED AS FOLLOWS: Article 1 Definitions For the purposes of this Framework Decision: 1. drugs: shall mean any of the substances covered by the following United Nations Conventions: (a) the 1961 Single Convention on Narcotic Drugs (as amended by the 1972 Protocol); (b) the 1971 Vienna Convention on Psychotropic Substances. It shall also include the substances subject to controls under Joint Action 97/396/JHA of 16 June 1997 concerning the information exchange risk assessment and the control of new synthetic drugs (4); 2. precursors: shall mean any substance scheduled in the Community legislation giving effect to the obligations deriving from Article 12 of the United Nations Convention against Illicit Traffic in Narcotic Drugs and Psychotropic Substances of 20 December 1988; 3. legal person: shall mean any legal entity having such status under the applicable national law, except for States or other public bodies acting in the exercise of their sovereign rights and for public international organisations. Article 2 Crimes linked to trafficking in drugs and precursors 1. Each Member State shall take the necessary measures to ensure that the following intentional conduct when committed without right is punishable: (a) the production, manufacture, extraction, preparation, offering, offering for sale, distribution, sale, delivery on any terms whatsoever, brokerage, dispatch, dispatch in transit, transport, importation or exportation of drugs; (b) the cultivation of opium poppy, coca bush or cannabis plant; (c) the possession or purchase of drugs with a view to conducting one of the activities listed in (a); (d) the manufacture, transport or distribution of precursors, knowing that they are to be used in or for the illicit production or manufacture of drugs. 2. The conduct described in paragraph 1 shall not be included in the scope of this Framework Decision when it is committed by its perpetrators exclusively for their own personal consumption as defined by national law. Article 3 Incitement, aiding and abetting and attempt 1. Each Member State shall take the necessary measures to make incitement to commit, aiding and abetting or attempting one of the offences referred to in Article 2 a criminal offence. 2. A Member State may exempt from criminal liability the attempt to offer or prepare drugs referred to in Article 2(1)(a) and the attempt to possess drugs referred to in Article 2(1)(c). Article 4 Penalties 1. Each Member State shall take the measures necessary to ensure that the offences defined in Articles 2 and 3 are punishable by effective, proportionate and dissuasive criminal penalties. Each Member State shall take the necessary measures to ensure that the offences referred to in Article 2 are punishable by criminal penalties of a maximum of at least between one and three years of imprisonment. 2. Each Member State shall take the necessary measures to ensure that the offences referred to in Article 2(1)(a), (b) and (c) are punishable by criminal penalties of a maximum of at least between 5 and 10 years of imprisonment in each of the following circumstances: (a) the offence involves large quantities of drugs; (b) the offence either involves those drugs which cause the most harm to health, or has resulted in significant damage to the health of a number of persons. 3. Each Member State shall take the necessary measures to ensure that the offences referred to in paragraph 2 are punishable by criminal penalties of a maximum of at least 10 years of deprivation of liberty, where the offence was committed within the framework of a criminal organisation as defined in Joint Action 98/733/JHA of 21 December 1998 on making it a criminal offence to participate in a criminal organisation in the Member States of the European Union (5). 4. Each Member State shall take the necessary measures to ensure that the offences referred to in Article 2(1)(d) are punishable by criminal penalties of a maximum of at least between 5 and 10 years of deprivation of liberty, where the offence was committed within the framework of a criminal organisation as defined in Joint Action 98/733/JHA and the precursors are intended to be used in or for the production or manufacture of drugs under the circumstances referred to in paragraphs 2(a) or (b). 5. Without prejudice to the rights of victims and of other bona fide third parties, each Member State shall take the necessary measures to enable the confiscation of substances which are the object of offences referred to in Articles 2 and 3, instrumentalities used or intended to be used for these offences and proceeds from these offences or the confiscation of property the value of which corresponds to that of such proceeds, substances or instrumentalities. The terms confiscation, instrumentalities, proceeds and property shall have the same meaning as in Article 1 of the 1990 Council of Europe Convention on Laundering, Search, Seizure and Confiscation of the Proceeds from Crime. Article 5 Particular circumstances Notwithstanding Article 4, each Member State may take the necessary measures to ensure that the penalties referred to in Article 4 may be reduced if the offender: (a) renounces criminal activity relating to trafficking in drugs and precursors, and (b) provides the administrative or judicial authorities with information which they would not otherwise have been able to obtain, helping them to: (i) prevent or mitigate the effects of the offence, (ii) identify or bring to justice the other offenders, (iii) find evidence, or (iv) prevent further offences referred to in Articles 2 and 3. Article 6 Liability of legal persons 1. Each Member State shall take the necessary measures to ensure that legal persons can be held liable for any of the criminal offences referred to in Articles 2 and 3 committed for their benefit by any person, acting either individually or as a member of an organ of the legal person in question, who has a leading position within the legal person, based on one of the following: (a) a power of representation of the legal person; (b) an authority to take decisions on behalf of the legal person; (c) an authority to exercise control within the legal person. 2. Apart from the cases provided for in paragraph 1, each Member State shall take the necessary measures to ensure that legal persons can be held liable where the lack of supervision or control by a person referred to in paragraph 1 has made possible the commission of any of the offences referred to in Articles 2 and 3 for the benefit of that legal person by a person under its authority. 3. Liability of legal persons under paragraphs 1 and 2 shall not exclude criminal proceedings against natural persons who are perpetrators, instigators or accessories in any of the offences referred to in Articles 2 and 3. Article 7 Sanctions for legal persons 1. Member States shall take the necessary measures to ensure that a legal person held liable pursuant to Article 6(1) is punishable by effective, proportionate and dissuasive sanctions, which shall include criminal or non-criminal fines and may include other sanctions, such as: (a) exclusion from entitlement to tax relief or other benefits or public aid; (b) temporary or permanent disqualification from the pursuit of commercial activities; (c) placing under judicial supervision; (d) a judicial winding-up order; (e) temporary or permanent closure of establishments used for committing the offence; (f) in accordance with Article 4(5), the confiscation of substances which are the object of offences referred to in Articles 2 and 3, instrumentalities used or intended to be used for these offences and proceeds from these offences or the confiscation of property the value of which corresponds to that of such proceeds, substances or instrumentalities. 2. Each Member State shall take the necessary measures to ensure that a legal person held liable pursuant to Article 6(2) is punishable by effective, proportionate and dissuasive sanctions or measures. Article 8 Jurisdiction and prosecution 1. Each Member State shall take the necessary measures to establish its jurisdiction over the offences referred to in Articles 2 and 3 where: (a) the offence is committed in whole or in part within its territory; (b) the offender is one of its nationals; or (c) the offence is committed for the benefit of a legal person established in the territory of that Member State. 2. A Member State may decide that it will not apply, or that it will apply only in specific cases or circumstances, the jurisdiction rules set out in paragraphs 1(b) and 1(c) where the offence is committed outside its territory. 3. A Member State which, under its laws, does not extradite its own nationals shall take the necessary measures to establish its jurisdiction over and to prosecute, where appropriate, an offence referred to in Articles 2 and 3 when it is committed by one of its own nationals outside its territory. 4. Member States shall inform the General Secretariat of the Council and the Commission when they decide to apply paragraph 2, where appropriate with an indication of the specific cases or circumstances in which the decision applies. Article 9 Implementation and reports 1. Member States shall take the necessary measures to comply with the provisions of this Framework Decision by 12 May 2006. 2. By the deadline referred to in paragraph 1, Member States shall transmit to the General Secretariat of the Council and to the Commission the text of the provisions transposing into their national law the obligations imposed on them under this Framework Decision. The Commission shall, by 12 May 2009, submit a report to the European Parliament and to the Council on the functioning of the implementation of the Framework Decision, including its effects on judicial cooperation in the field of illicit drug trafficking. Following this report, the Council shall assess, at the latest within six months after submission of the report, whether Member States have taken the necessary measures to comply with this Framework Decision. Article 10 Territorial application This Framework Decision shall apply to Gibraltar. Article 11 Entry into force This Framework Decision shall enter into force on the day following its publication in the Official Journal of the European Union. Done at Luxembourg, 25 October 2004. For the Council The President R. VERDONK (1) OJ C 304 E, 30.10.2001, p. 172. (2) Opinion of 9 March 2004 (not yet published in the Official Journal). (3) OJ C 19, 23.1.1999, p. 1. (4) OJ L 167, 25.6.1997, p. 1. (5) OJ L 351, 29.12.1998, p. 1.
============================== "END OF DOC" ==============================
        

        doc 3 :

        'celex': 32013R1382
        'status': In Force
        'act_type': Regulation
        'treaty': TFEU (2008)

        full_doc :

        28.12.2013 EN Official Journal of the European Union L 354/73 REGULATION (EU) No 1382/2013 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 17 December 2013 establishing a Justice Programme for the period 2014 to 2020 (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 81(1) and (2), Article 82(1) and Article 84 thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), Having regard to the opinion of the Committee of the Regions (2), Acting in accordance with the ordinary legislative procedure (3), Whereas: (1) The Treaty on the Functioning of the European Union (TFEU) provides for the creation of an area of freedom, security and justice, in which persons are free to move. To that end, the Union may adopt measures to develop judicial cooperation in civil and criminal matters and to promote and support the action of Member States in the field of crime prevention. Respect for fundamental rights as well as for common principles, such as non-discrimination, gender equality, effective access to justice for all, the rule of law and a well-functioning independent judicial system should be ensured in the further development of a European area of justice. (2) In the Stockholm Programme (4) the European Council reaffirmed the priority of developing an area of freedom, security and justice and specified as a political priority the achievement of a Europe of law and justice. Financing was identified as one of the important tools for the successful implementation of the Stockholm Programme's political priorities. The ambitious goals set by the Treaties and by the Stockholm Programme should be attained inter alia by establishing, for the period 2014 to 2020, a flexible and effective Justice Programme (the "Programme") which should facilitate planning and implementation. The general and specific objectives of the Programme should be interpreted in line with the relevant strategic guidelines defined by the European Council. (3) The Commission Communication of 3 March 2010 on the Europe 2020 Strategy sets out a strategy for smart, sustainable and inclusive growth. A well-functioning area of justice, where obstacles in cross-border judicial proceedings and access to justice in cross-border situations are eliminated, should be developed as a key element to support the specific objectives and flagship initiatives of the Europe 2020 Strategy and to facilitate mechanisms designed to promote growth. (4) For the purposes of this Regulation, the term "judiciary and judicial staff" should be interpreted so as to include judges, prosecutors and court officers, as well as other legal practitioners associated with the judiciary, such as lawyers, notaries, bailiffs, probation officers, mediators and court interpreters. (5) Judicial training is central to building mutual trust and improves cooperation between judicial authorities and practitioners in the various Member States. Judicial training should be seen as an essential element in promoting a genuine European judicial culture in the context of the Commission Communication of 13 September 2011 entitled "Building trust in EU-wide justice. A new dimension to European judicial training", the Council Resolution on the training of judges, prosecutors and judicial staff in the European Union (5), the Council conclusions of 27 and 28 October 2011 on European judicial training and the European Parliament resolution of 14 March 2012 on judicial training. (6) Judicial training can involve different actors, such as Member States' legal, judicial and administrative authorities, academic institutions, national bodies responsible for judicial training, European-level training organisations or networks, or networks of court coordinators of Union law. Bodies and entities pursuing a general European interest in the field of training of the judiciary, such as the European Judicial Training Network (EJTN), the Academy of European Law (ERA), the European Network of Councils for the Judiciary (ENCJ), the Association of the Councils of State and Supreme Administrative Jurisdictions of the European Union (ACA-Europe), the Network of the Presidents of Supreme Judicial Courts of the European Union (RPCSJUE) and the European Institute of Public Administration (EIPA), should continue to play their role in promoting training programmes with a genuine European dimension for the judiciary and judicial staff, and could therefore be granted adequate financial support in accordance with the procedures and the criteria set out in the annual work programmes adopted by the Commission pursuant to this Regulation. (7) The Union should facilitate training activities on the implementation of Union law by considering the salaries of participating judiciary and judicial staff incurred by the Member States' authorities as eligible costs or co-financing in kind, in accordance with Regulation (EU, Euratom) No 966/2012 of the European Parliament and of the Council (6) (the "Financial Regulation"). (8) Access to justice should include, in particular, access to courts, to alternative methods of dispute settlement and to public office-holders obliged by the law to provide parties with independent and impartial legal advice. (9) In December 2012 the Council endorsed the EU Drugs Strategy (2013-20) (7), which aims to take a balanced approach based on simultaneous reduction of drug demand and drug supply, acknowledging that drug demand reduction and drug supply reduction are mutually reinforcing elements in illicit drugs policy. That Strategy maintains as one of its main objectives the aim of contributing to a measurable reduction of drug demand, of drug dependence and of drug-related health and social risks and harms. Whereas the Drug prevention and information programme established by Decision No 1150/2007/EC of the European Parliament and of the Council (8) was based on a public health legal basis and covered those aspects, the Programme is founded on a different legal basis and should aim at the further development of a European area of justice based on mutual recognition and mutual trust, in particular by promoting judicial cooperation. Thus, in responding to the need for simplification and in line with the legal basis of each programme, the Health for Growth Programme can support measures to complement the Member's States action in attaining the objective of reducing drug-related health damage, including information and prevention. (10) Another important element of the EU Drugs Strategy (2013-20) is drug supply reduction. Whereas the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, should support actions aimed at preventing and combating the trafficking of drugs and other types of crime, and in particular measures targeting the production, manufacture, extraction, sale, transport, importation and exportation of illegal drugs, including possession and purchase with a view to engaging in drug trafficking activities, the Programme should cover those aspects of drugs policy that are not covered by the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, or by the Health for Growth Programme and are closely linked to its general objective. (11) In any case, the continued financing of the priorities under the 2007-2013 programming period that have been maintained as objectives under the new EU Drugs Strategy (2013-20) should be ensured, and funds should therefore be available from the Health for Growth Programme, the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, and the Programme in accordance with their respective priorities and legal bases while avoiding any duplicate financing. (12) Pursuant to Article 3(3) of the Treaty on European Union (TEU), Article 24 of the Charter of Fundamental Rights of the European Union (the "Charter") and the 1989 United Nations Convention on the Rights of the Child, the Programme should support the protection of the rights of the child, including the right to due process, the right to understand the proceedings, the right to respect for private and family life and the right to integrity and dignity. The Programme should aim, in particular, to increase child protection within justice systems and access to justice for children, and should mainstream the promotion of the rights of the child in the implementation of all of its actions. (13) Pursuant to Articles 8 and 10 TFEU, the Programme should support the mainstreaming of equality between women and men and non-discrimination objectives in all its activities. Regular monitoring and evaluation should be carried out to assess the way in which gender equality and non-discrimination issues are addressed in the Programme's activities. (14) Experience of action at Union level has shown that achieving the objectives of the Programme in practice calls for a combination of instruments, including legal acts, policy initiatives and funding. Funding is an important tool complementing legislative measures. (15) In its conclusions of 22 and 23 September 2011 on improving the efficiency of future Union financial programmes supporting judicial cooperation, the Council stressed the important role played by Union financing programmes in the efficient implementation of the Union acquis and reiterated the need for more transparent, flexible, coherent and streamlined access to those programmes. (16) The Commission Communication of 29 June 2011 entitled 'A budget for Europe 2020' stresses the need for the rationalisation and simplification of Union funding. Especially in view of the current economic crisis, it is of the utmost importance that Union funds be structured and managed in the most diligent manner. Meaningful simplification and efficient management of funding can be achieved through a reduction in the number of programmes and through the rationalisation, simplification and harmonisation of funding rules and procedures. (17) In responding to the need for simplification, efficient management and easier access to funding, the Programme should continue and develop activities previously carried out on the basis of three programmes established by Council Decision 2007/126/JHA (9), Decision No 1149/2007/EC of the European Parliament and of the Council (10), and Decision No 1150/2007/EC. The mid-term evaluations of those programmes include recommendations aimed at improving the implementation of those programmes. The findings of those mid-term evaluations, as well as the findings of the respective ex-post evaluations, need to be taken into account in the implementation of the Programme. (18) The Commission Communication of 19 October 2010 entitled 'The EU Budget Review' and the Commission Communication of 29 June 2011 entitled 'A budget for Europe 2020' underline the importance of focusing funding on activities with clear European added value, i.e. where Union intervention can bring additional value compared to the action of Member States alone. Actions covered by this Regulation should contribute to the creation of a European area of justice by promoting the principle of mutual recognition, developing mutual trust between the Member States, increasing cross-border cooperation and networking and achieving the correct, coherent and consistent application of Union law. Funding activities should also contribute to achieving effective and better knowledge of Union law and policies by all concerned, and should provide a sound analytical basis for the support and the development of Union law and policies, in so doing contributing to their enforcement and proper implementation. Union intervention allows for those actions to be pursued consistently across the Union and brings economies of scale. Moreover, the Union is in a better position than Member States to address cross-border situations and to provide a European platform for mutual learning. (19) In selecting actions for funding under the Programme, the Commission should assess the proposals against pre-identified criteria. Those criteria should include an assessment of the European added value of the proposed actions. National projects and small-scale projects can also have European added value. (20) Bodies and entities that have access to the Programme should include national, regional and local authorities. (21) This Regulation lays down a financial envelope for the entire duration of the Programme which is to constitute the prime reference amount, within the meaning of point 17 of the Interinstitutional Agreement of 2 December 2013 between the European Parliament, the Council and the Commission on budgetary discipline, on cooperation in budgetary matters and on sound financial management (11), for the European Parliament and the Council during the annual budgetary procedure. (22) In order to ensure that the Programme is sufficiently flexible to respond to changing needs and corresponding policy priorities throughout its duration, the power to adopt acts in accordance with Article 290 TFEU should be delegated to the Commission concerning modification of the percentages set out in the Annex to this Regulation for each specific objective that would exceed those percentages by more than 5 percentage points. To assess the need for such a delegated act, those percentages should be calculated on the basis of the financial envelope of the Programme for its entire duration, and not on the basis of annual appropriations. It is of particular importance that the Commission carry out appropriate consultations during its preparatory work, including at expert level. The Commission, when preparing and drawing up delegated acts, should ensure a simultaneous, timely and appropriate transmission of relevant documents to the European Parliament and to the Council. (23) This Regulation should be implemented in full compliance with the Financial Regulation. In particular with regard to the eligibility conditions of value added tax (VAT) paid by grant beneficiaries, the eligibility of VAT should not depend on the legal status of the beneficiaries for activities which can be carried out by private and public bodies and entities under the same legal conditions. Taking into account the specific nature of the objectives and activities covered by this Regulation, it should be made clear, in calls for proposals, that, for activities which can be carried out by both public and private bodies and entities, the non-deductible VAT incurred by public bodies and entities is to be eligible, in so far as it is paid in respect of the implementation of activities, such as training or awareness-raising, which cannot be considered as the exercise of public authority. This Regulation should also make use of the simplification tools introduced by the Financial Regulation. Moreover, the criteria for identifying actions to be supported should aim at allocating the available financial resources to actions generating the highest impact in relation to the policy objective pursued. (24) In order to ensure uniform conditions for the implementation of this Regulation, implementing powers should be conferred on the Commission in respect of the adoption of annual work programmes. Those powers should be exercised in accordance with Regulation (EU) No 182/2011 of the European Parliament and of the Council (12). (25) The annual work programmes adopted by the Commission pursuant to this Regulation should ensure appropriate distribution of funds between grants and public procurement contracts. The Programme should primarily allocate funds to grants, while maintaining sufficient funding levels for procurement. The minimum percentage of annual expenditure to be allocated to grants should be established in the annual work programmes and should be not less than 65 %. To facilitate project planning and co-financing by stakeholders, the Commission should establish a clear timetable for the calls for proposals, selection of projects and award decisions. (26) In order to ensure efficient allocation of funds from the general budget of the Union, consistency, complementarity and synergies should be sought between funding programmes supporting policy areas with close links to each other, in particular between the Programme and the Rights, Equality and Citizenship Programme established by Regulation (EU) No 1381/2013 of the European Parliament and of the Council (13), the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, the Health for Growth Programme, the Erasmus+ Programme established by Regulation (EU) No 1288/2013 of the European Parliament and of the Council (14), the Horizon 2020 Framework Programme established by Regulation (EU) No 1291/2013 of the European Parliament and of the Council (15) and the Instrument for Pre-accession Assistance (IPA II). (27) The financial interests of the Union should be protected through proportionate measures throughout the expenditure cycle, including the prevention, detection and investigation of irregularities, the recovery of funds lost, wrongly paid or incorrectly used and, where appropriate, the imposition of administrative and financial penalties in accordance with the Financial Regulation. (28) In order to implement the principle of sound financial management, this Regulation should provide for appropriate tools to assess its performance. To that end, it should define general and specific objectives. To measure the achievement of those specific objectives, a set of concrete and quantifiable indicators should be established which should remain valid for the whole duration of the Programme. The Commission should submit annually to the European Parliament and to the Council a monitoring report which should be based inter alia on the indicators set out in this Regulation and which should give information on the use of available funds. (29) The Programme should be implemented in an effective manner, respecting sound financial management, while also allowing potential applicants to have effective access to the Programme. In order to support effective access to the Programme, the Commission should use its best endeavours to simplify and harmonise the application procedures and documents, the administrative formalities and the financial management requirements, to remove administrative burdens and to encourage grant applications from entities located in Member States which are under-represented in the Programme. The Commission should publish on a dedicated webpage information about the Programme, its objectives, the various calls for proposals and their time schedules. Basic documents and guidelines relating to the calls for proposals should be available in all the official languages of the institutions of the Union. (30) In accordance with point (l) of Article 180(1) of Commission Delegated Regulation (EU) No 1268/2012 (16) ('the Rules of Application'), the grant agreements should lay down provisions governing the visibility of the Union financial support, except in duly justified cases where public display is not possible or appropriate. (31) In accordance with Article 35(2) and (3) of the Financial Regulation and Article 21 of the Rules of Application, the Commission should make available, in an appropriate and timely manner, information concerning recipients and concerning the nature and purpose of the measures financed from the general budget of the Union. That information should be made available with due observance of the requirements of confidentiality and security, in particular the protection of personal data. (32) Since the objective of this Regulation, namely to contribute to the further development of a European area of justice based on mutual recognition and mutual trust, in particular by promoting judicial cooperation in civil and criminal matters, cannot be sufficiently achieved by the Member States but can rather, by reason of its scale and effects, be better achieved at Union level, the Union may adopt measures, in accordance with the principle of subsidiarity as set out in Article 5 TEU. In accordance with the principle of proportionality, as set out in that Article, this Regulation does not go beyond what is necessary in order to achieve that objective. (33) In accordance with Article 3 of Protocol No 21 on the position of the United Kingdom and Ireland in respect of the Area of Freedom, Security and Justice, annexed to the TEU and to the TFEU, Ireland has notified its wish to take part in the adoption and application of this Regulation. (34) In accordance with Articles 1 and 2 of Protocol No 21 on the position of the United Kingdom and Ireland in respect of the Area of Freedom, Security and Justice, annexed to the TEU and to the TFEU, and without prejudice to Article 4 of that Protocol, the United Kingdom is not taking part in the adoption of this Regulation and is not bound by it or subject to its application. (35) In accordance with Articles 1 and 2 of Protocol No 22 on the position of Denmark, annexed to the TEU and to the TFEU, Denmark is not taking part in the adoption of this Regulation and is not bound by it or subject to its application. (36) In order to ensure the continuity of funding of activities previously carried out on the basis of Decision 2007/126/JHA, Decision No 1149/2007/EC and Decision No 1150/2007/EC, this Regulation should enter into force on the day following that of its publication, HAVE ADOPTED THIS REGULATION: Article 1 Establishment and duration of the Programme 1. This Regulation establishes a Justice programme ('the Programme'). 2. The Programme shall cover the period from 1 January 2014 to 31 December 2020. Article 2 European added value 1. The Programme shall finance actions with European added value which contribute to the further development of a European area of justice. To that end, the Commission shall ensure that the actions selected for funding are intended to produce results with European added value. 2. The European added value of actions, including that of small-scale and national actions, shall be assessed in the light of criteria such as their contribution to the consistent and coherent implementation of Union law and to wide public awareness about the rights deriving from it, their potential to develop mutual trust among Member States and to improve cross-border cooperation, their transnational impact, their contribution to the elaboration and dissemination of best practices or their potential to create practical tools and solutions that address cross-border or Union-wide challenges. Article 3 General objective The general objective of the Programme shall be to contribute to the further development of a European area of justice based on mutual recognition and mutual trust, in particular by promoting judicial cooperation in civil and criminal matters. Article 4 Specific objectives 1. To achieve the general objective set out in Article 3, the Programme shall have the following specific objectives: (a) to facilitate and support judicial cooperation in civil and criminal matters; (b) to support and promote judicial training, including language training on legal terminology, with a view to fostering a common legal and judicial culture; (c) to facilitate effective access to justice for all, including to promote and support the rights of victims of crime, while respecting the rights of the defence; (d) to support initiatives in the field of drugs policy as regards judicial cooperation and crime prevention aspects closely linked to the general objective of the Programme, in so far as they are not covered by the Internal security fund for financial support for police cooperation, preventing and combating crime, and crisis management or by the Health for Growth Programme; 2. The specific objectives of the Programme shall be pursued through, in particular: (a) enhancing public awareness and knowledge of Union law and policies; (b) with a view to ensuring efficient judicial cooperation in civil and criminal matters, improving knowledge of Union law, including substantive and procedural law, of judicial cooperation instruments and of the relevant case-law of the Court of Justice of the European Union, and of comparative law; (c) supporting the effective, comprehensive and consistent implementation and application of Union instruments in the Member States and the monitoring and evaluation thereof; (d) promoting cross-border cooperation, improving mutual knowledge and understanding of the civil and criminal law and the legal and judicial systems of the Member States and enhancing mutual trust; (e) improving knowledge and understanding of potential obstacles to the smooth functioning of a European area of justice; (f) improving the efficiency of judicial systems and their cooperation by means of information and communication technology, including the cross-border interoperability of systems and applications. Article 5 Mainstreaming In the implementation of all of its actions, the Programme shall seek to promote equality between women and men and to promote the rights of the child, inter alia by means of child-friendly justice. It shall also comply with the prohibition of discrimination based on any of the grounds listed in Article 21 of the Charter, in accordance with and within the limits set by Article 51 of the Charter. Article 6 Types of actions 1. The Programme shall finance inter alia the following types of actions: (a) analytical activities, such as the collection of data and statistics; the development of common methodologies and, where appropriate, indicators or benchmarks; studies, researches, analyses and surveys; evaluations; the elaboration and publication of guides, reports and educational material; workshops, seminars, experts meetings and conferences; (b) training activities, such as staff exchanges, workshops, seminars, train-the-trainer events, including language training on legal terminology, and the development of online training tools or other training modules for members of the judiciary and judicial staff; (c) mutual learning, cooperation, awareness-raising and dissemination activities, such as the identification of, and exchanges concerning, good practices, innovative approaches and experiences; the organisation of peer reviews and mutual learning; the organisation of conferences, seminars, information campaigns, including institutional communication on the political priorities of the Union as far as they relate to the objectives of the Programme; the compilation and publication of materials to disseminate information about the Programme and its results; the development, operation and maintenance of systems and tools, using information and communication technologies, including the further development of the European e-Justice portal as a tool to improve citizens' access to justice; (d) support for main actors whose activities contribute to the implementation of the objectives of the Programme, such as support for Member States in the implementation of Union law and policies, support for key European actors and European-level networks, including in the field of judicial training; and support for networking activities at European level among specialised bodies and entities as well as national, regional and local authorities and non-governmental organisations. 2. The European Judicial Training Network shall receive an operating grant to co-finance expenditure associated with its permanent work programme. Article 7 Participation 1. Access to the Programme shall be open to all bodies and entities legally established in: (a) Member States; (b) European Free Trade Association (EFTA) countries which are parties to the Agreement on the European Economic Area, in accordance with that Agreement; (c) candidate countries, potential candidates and countries acceding to the Union, in accordance with the general principles and the general terms and conditions laid down for the participation of those countries in the Union programmes established in the respective Framework Agreements and Association Council decisions, or similar agreements. 2. Bodies and entities which are profit-oriented shall have access to the Programme only in conjunction with non-profit or public organisations. 3. Bodies and entities legally established in third countries, other than those participating in the Programme in accordance with points (b) and (c) of paragraph 1, in particular countries where the European Neighbourhood Policy applies, may be associated to the actions of the Programme at their own cost, if this serves the purpose of those actions. 4. The Commission may cooperate with international organisations under the conditions laid down in the relevant annual work programme. Access to the Programme shall be open to international organisations active in the areas covered by the Programme in accordance with the Financial Regulation and the relevant annual work programme. Article 8 Budget 1. The financial envelope for the implementation of the Programme for the period 2014 to 2020 is set at EUR 377 604 000. 2. The financial allocation of the Programme may also cover expenses pertaining to preparatory, monitoring, control, audit and evaluation activities which are required for the management of the Programme and the assessment of the achievement of its objectives. The financial allocation may cover expenses relating to the necessary studies, meetings of experts, information and communication actions, including institutional communication of the political priorities of the Union, in so far as they are related to the general objectives of this Regulation, as well as expenses linked to information technology networks focusing on information processing and exchange and other technical and administrative assistance needed in connection with the management of the Programme by the Commission. 3. The annual appropriations shall be authorised by the European Parliament and the Council within the limits of the multiannual financial framework established by Council Regulation (EU, Euratom) No 1311/2013 (17). 4. Within the financial envelope for the Programme, amounts shall be allocated to each specific objective in accordance with the percentages set out in the Annex. 5. The Commission shall not depart from the allocated percentages of the financial envelope, as set out in the Annex, by more than 5 percentage points for each specific objective. Should it prove necessary to exceed that limit, the Commission shall be empowered to adopt delegated acts in accordance with Article 9 to modify each of the figures in the Annex by more than 5 and up to 10 percentage points. Article 9 Exercise of the delegation 1. The power to adopt delegated acts is conferred on the Commission subject to the conditions laid down in this Article. 2. The power to adopt delegated acts referred to in Article 8(5) shall be conferred on the Commission for the duration of the Programme. 3. The delegation of power referred to in Article 8(5) may be revoked at any time by the European Parliament or by the Council. A decision to revoke shall put an end to the delegation of the power specified in that decision. It shall take effect the day following the publication of the decision in the Official Journal of the European Union or at a later date specified therein. It shall not affect the validity of any delegated acts already in force. 4. As soon as it adopts a delegated act, the Commission shall notify it simultaneously to the European Parliament and to the Council. 5. A delegated act adopted pursuant to Article 8(5) shall enter into force only if no objection has been expressed either by the European Parliament or the Council within a period of two months of notification of that act to the European Parliament and the Council or if, before the expiry of that period, the European Parliament and the Council have both informed the Commission that they will not object. That period shall be extended by two months at the initiative of the European Parliament or the Council. Article 10 Implementing measures 1. The Commission shall implement the Programme in accordance with the Financial Regulation. 2. In order to implement the Programme, the Commission shall adopt annual work programmes in the form of implementing acts. Those implementing acts shall be adopted in accordance with the examination procedure referred to in Article 11(2). 3. Each annual work programme shall implement the objectives of the Programme by determining the following: (a) the actions to be undertaken, in accordance with the general and specific objectives set out in Article 3 and Article 4(1), including the indicative allocation of financial resources; (b) the essential eligibility, selection and award criteria to be used to select the proposals which are to receive financial contributions, in accordance with Article 84 of the Financial Regulation and with Article 94 of its Rules of Application; (c) the minimum percentage of annual expenditure to be allocated to grants. 4. Appropriate and fair distribution of financial support between different areas covered by this Regulation shall be ensured. When deciding on the allocation of funds to those areas in the annual work programmes, the Commission shall take into consideration the need to maintain sufficient funding levels for both civil justice and criminal justice, as well as for judicial training and initiatives in the field of drugs policy within the scope of the Programme. 5. Calls for proposals shall be published on an annual basis. 6. In order to facilitate judicial training activities, the costs associated with the participation of judiciary and judicial staff in those activities and incurred by the Member States' authorities shall be taken into account in accordance with the Financial Regulation when providing corresponding funding. Article 11 Committee procedure 1. The Commission shall be assisted by a committee. That committee shall be a committee within the meaning of Regulation (EU) No 182/2011. 2. Where reference is made to this paragraph, Article 5 of Regulation (EU) No 182/2011 shall apply. Article 12 Complementarity 1. The Commission, in cooperation with the Member States, shall ensure overall consistency, complementarity and synergies with other Union instruments including, inter alia, the Rights, Equality and Citizenship Programme, the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, the Health for Growth Programme, the Erasmus+ Programme, the Horizon 2020 Framework Programme and the Instrument for Pre-accession Assistance (IPA II). 2. The Commission shall also ensure overall consistency, complementarity and synergies with the work of the Union bodies, offices and agencies operating in areas covered by the objectives of the Programme, such as Eurojust established by Council Decision 2002/187/JHA (18) and the European Monitoring Centre for Drugs and Drug Addiction (EMCDDA) established by Regulation (EC) No 1920/2006 of the European Parliament and of the Council (19). 3. The Programme may share resources with other Union instruments, in particular the Rights, Equality and Citizenship Programme, in order to implement actions meeting the objectives of both programmes. An action for which funding has been awarded from the Programme may also give rise to the award of funding from the Rights, Equality and Citizenship Programme, provided that the funding does not cover the same cost items. Article 13 Protection of the financial interests of the Union 1. The Commission shall take appropriate measures ensuring that, when actions financed under the Programme are implemented, the financial interests of the Union are protected by the application of preventive measures against fraud, corruption and any other illegal activities, by effective checks and, if irregularities are detected, by the recovery of amounts wrongly paid and, where appropriate, by effective, proportionate and dissuasive administrative and financial penalties. 2. The Commission or its representatives and the Court of Auditors shall have the power of audit, both on the basis of documents and on the spot, over all grant beneficiaries, contractors and subcontractors who have received Union funds under the Programme. 3. The European Anti-Fraud Office (OLAF) may carry out investigations, including on-the-spot checks and inspections, in accordance with the provisions and procedures laid down in Regulation (EU, Euratom) No 883/2013 of the European Parliament and of the Council (20) and in Council Regulation (Euratom, EC) No 2185/96 (21) with a view to establishing whether fraud, corruption or any other illegal activity has occurred affecting the financial interests of the Union in connection with a grant agreement or grant decision or a contract funded under the Programme. 4. Without prejudice to paragraphs 1, 2 and 3, cooperation agreements with third countries and with international organisations, grant agreements, grant decisions and contracts resulting from the implementation of the Programme shall contain provisions expressly empowering the Commission, the Court of Auditors and OLAF to conduct the audits and investigations referred to in those paragraphs, in accordance with their respective competences. Article 14 Monitoring and evaluation 1. The Commission shall monitor the Programme annually in order to follow the implementation of actions carried out under it and the achievement of the specific objectives set out in Article 4. The monitoring shall also provide a means of assessing the way in which gender equality and non-discrimination issues have been addressed across the Programme's actions. 2. The Commission shall provide the European Parliament and the Council with: (a) an annual monitoring report based on the indicators set out in Article 15(2) and on the use of the available funds; (b) an interim evaluation report by 30 June 2018; (c) an ex-post evaluation report by 31 December 2021. 3. The interim evaluation report shall assess the achievement of the Programme's objectives, the efficiency of the use of resources and the Programme's European added value with a view to determining whether funding in areas covered by the Programme should be renewed, modified or suspended after 2020. It shall also address the scope for any simplification of the Programme, its internal and external coherence, and the continued relevance of all objectives and actions. It shall take into account the results of the ex-post evaluations of the previous 2007-2013 programmes established by the Decisions referred to in Article 16. 4. The ex-post evaluation report shall assess the long-term impact of the Programme and the sustainability of the effects of the Programme, with a view to informing a decision on a subsequent programme. 5. The evaluations shall also assess the way in which gender equality and non-discrimination issues have been addressed across the Programme's actions. Article 15 Indicators 1. In accordance with Article 14, the indicators set out in paragraph 2 of this Article shall serve as a basis for monitoring and evaluating the extent to which each of the Programme's specific objectives set out in Article 4 has been achieved through the actions provided for in Article 6. They shall be measured against pre-defined baselines reflecting the situation before implementation. Where relevant, indicators shall be broken down by, inter alia, sex, age and disability. 2. The indicators referred to in paragraph 1 shall include, inter alia, the following: (a) the number and percentage of persons in a target group reached by awareness-raising activities funded by the Programme; (b) the number and percentage of members of the judiciary and judicial staff in a target group that participated in training activities, staff exchanges, study visits, workshops and seminars funded by the Programme; (c) the improvement in the level of knowledge of Union law and policies in the groups participating in activities funded by the Programme compared to the entire target group; (d) the number of cases, activities and outputs of cross-border cooperation, including cooperation by means of information technology tools and procedures established at Union level; (e) participants' assessment of the activities in which they participated and of their (expected) sustainability; (f) the geographical coverage of the activities funded by the Programme. 3. In addition to the indicators set out in paragraph 2, the interim and ex-post evaluation report of the Programme shall assess, inter alia: (a) the perceived impact of the Programme on access to justice based on qualitative and quantitative data collected at European level; (b) the number and quality of instruments and tools developed through actions funded by the Programme; (c) the European added value of the Programme, including an evaluation of the Programme's activities in the light of similar initiatives which have been developed at national or European level without support from Union funding, and their (expected) results and the advantages and/or disadvantages of Union funding compared to national funding for the type of activity in question; (d) the level of funding in relation to the outcomes achieved (efficiency); (e) the possible administrative, organisational and/or structural obstacles to the smoother, more effective and efficient implementation of the Programme (scope for simplification). Article 16 Transitional measures Actions initiated on the basis of Decision 2007/126/JHA, Decision 1149/2007/EC or Decision 1150/2007/EC shall continue to be governed by the provisions of those Decisions until their completion. In respect of those actions, reference to the committees provided for in Article 9 of Decision 2007/126/JHA, in Articles 10 and 11 of Decision 1149/2007/EC and in Article 10 of Decision 1150/2007/EC shall be interpreted as references to the committee provided for in Article 11(1) of this Regulation. Article 17 Entry into force This Regulation shall enter into force on the day following that of its publication in the Official Journal of the European Union. This Regulation shall be binding in its entirety and directly applicable in the Member States in accordance with the Treaties. Done at Brussels, 17 December 2013. For the European Parliament The President M. SCHULZ For the Council The President L. LINKEVIÃ IUS (1) OJ C 299, 4.10.2012, p. 103. (2) OJ C 277, 13.9.2012, p. 43. (3) Position of the European Parliament of 11 December 2013 (not yet published in the Official Journal) and decision of the Council of 16 December. (4) OJ C 115, 4.5.2010, p. 1. (5) OJ C 299, 22.11.2008, p. 1. (6) Regulation (EU, Euratom) No 966/2012 of the European Parliament and of the Council of 25 October 2012 on the financial rules applicable to the general budget of the Union and repealing Council Regulation (EC, Euratom) No 1605/2002 (OJ L 298, 26.10.2012, p. 1). (7) OJ C 402, 29.12.2012, p. 1. (8) Decision No 1150/2007/EC of the European Parliament and of the Council of 25 September 2007 establishing for the period 2007-2013 the Specific Programme 'Drug prevention and information' as part of the General Programme 'Fundamental Rights and Justice' (OJ L 257, 3.10.2007, p. 23). (9) Council Decision 2007/126/JHA of 12 February 2007 establishing for the period 2007-2013, as part of the General Programme on Fundamental Rights and Justice, the Specific Programme 'Criminal Justice' (OJ L 58, 24.2.2007, p. 13). (10) Decision No 1149/2007/EC of the European Parliament and of the Council of 25 September 2007 establishing for the period 2007-2013 the Specific Programme 'Civil Justice' as part of the General Programme 'Fundamental Rights and Justice (OJ L 257, 3.10.2007, p. 16). (11) OJ C 373, 20.12.2013, p. 1. (12) Regulation (EU) No 182/2011 of the European Parliament and of the Council of 16 February 2011 laying down the rules and general principles concerning mechanisms for control by Member States of the Commission's exercise of implementing powers (OJ L 55, 28.2.2011, p. 13). (13) Regulation (EU) No 1381/2013 of the European Parliament and of the Council of 17 December 2013 establishing a Rights, Equality and Citizenship Programme for the period 2014 to 2020 (see page 62 of this Official Journal). (14) Regulation (EU) No 1288/2013 of the European Parliament and of the Council of 11 December 2013 establishing "Erasmus+": the Union programme for education, training, youth and sport and repealing Decisions No 1719/2006/EC, No 1720/2006/EC and No 1298/2008/EC (OJ L 347, 20.12.2013, p. 50). (15) Regulation (EU) No 1291/2013 of the European Parliament and of the Council of 11 December 2013 establishing Horizon 2020 - the Framework Programme for Research and Innovation (2014-2020) and repealing Decision No 1982/2006/EC (OJ L 347, 20.12.2013, p. 104). (16) Commission Delegated Regulation (EU) No 1268/2012 of 29 October 2012 on the rules of application of Regulation (EU, Euratom) No 966/2012 of the European Parliament and of the Council on the financial rules applicable to the general budget of the Union (OJ L 362, 31.12.2012, p. 1). (17) Council Regulation (EU, Euratom) No 1311/2013 of 2 December 2013 laying down the multiannual financial framework for the years 2014-2020 (OJ L 347, 20.12.2013, p. 884). (18) Council Decision 2002/187/JHA of 28 February 2002 setting up Eurojust with a view to reinforcing the fight against serious crime (OJ L 63, 6.3.2002, p. 1). (19) Regulation (EC) No 1920/2006 of the European Parliament and of the Council of 12 December 2006 on the European Monitoring Centre for Drugs and Drug Addiction (OJ L 376, 27.12.2006, p. 1). (20) Regulation (EU, Euratom) No 883/2013 of the European Parliament and of the Council of 11 September 2013 concerning investigations conducted by the European Anti-Fraud Office (OLAF) and repealing Regulation (EC) No 1073/1999 of the European Parliament and of the Council and Council Regulation (Euratom) No 1074/1999 (OJ L 248, 18.9.2013, p. 1). (21) Council Regulation (Euratom, EC) No 2185/96 of 11 November 1996 concerning on-the-spot checks and inspections carried out by the Commission in order to protect the European Communities' financial interests against fraud and other irregularities (OJ L 292, 15.11.1996, p. 2). ANNEX ALLOCATION OF FUNDS Within the financial envelope for the Programme, amounts shall be allocated as follows to each specific objective set out in Article 4(1): Specific objectives Share of the financial envelope (in %) (a) to facilitate and support judicial cooperation in civil and criminal matters 30 % (b) to support and promote judicial training, including language training on legal terminology, with a view to fostering a common legal and judicial culture 35 % (c) to facilitate effective access to justice for all, including to promote and support the rights of victims of crime, while respecting the rights of the defence 30 % (d) to support initiatives in the field of drugs policy as regards judicial cooperation and crime prevention aspects closely linked to the general objective of the Programme, in so far as they are not covered by the Instrument for financial support for police cooperation, preventing and combating crime, and crisis management, as part of the Internal Security Fund, or by the Health for Growth Programme 5 %.
============================== "END OF DOC" ==============================
        

In [6]:
import pandas as pd

In [7]:
laws = pd.read_csv('dataset/act_raw_text_with_4meta.csv')

In [17]:
laws.columns


Index(['Unnamed: 0', 'CELEX', 'Status', 'Act_type', 'Treaty', 'act_raw_text'], dtype='str')

In [8]:
def get__embeddings(enhanced_query):
    query_embedding = embedding_model.embed_query(enhanced_query)

    return query_embedding

In [9]:
query_embedding = get__embeddings("driving without license penalty")

In [10]:
weaviate_client.is_live()

True

In [28]:
eu = weaviate_client.collections.use("Euro_Laws")
response = eu.query.near_vector(
    near_vector= query_embedding, 
    limit=5
)

for obj in response.objects:
    print(json.dumps(obj.properties, indent=10))  # Inspect the results

KeyboardInterrupt: 

In [11]:
vectorstore = WeaviateVectorStore(
    client = weaviate_client,
    index_name = "Euro_Laws",
    text_key="text",
    embedding = embedding_model
)

In [26]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("driving without license penalty")
docs

ValueError: Error during query: Query call with protocol GRPC search failed with message Deadline Exceeded.

```
'celex': , 'status': 

'act_type' , 'treaty':

```

In [15]:
def get_celex_ids(docs):
    celex_ids = []
    for i in docs:
        celex_ids.append(i.metadata['celex'])

    celex_ids = list(set(celex_ids))

    return celex_ids

get_celex_ids(docs)  

['32015L0413', '32006L0126', '32009R1072', '31980L1263']

In [22]:
laws[laws['CELEX'] == '32015L0413']['Status'].iloc[0]

'In Force'

In [18]:
def search_docs(query):
    celex_ids = []
    full_doc_info = """ """

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list(set(celex_ids))    

    for i , celex_id in enumerate(celex_ids):

        full_doc_info += f"""

        doc {i} :

        'celex': {laws[laws['CELEX'] == celex_id]['CELEX'].iloc[0]}
        'status': {laws[laws['CELEX'] == celex_id]['Status'].iloc[0]}
        'act_type': {laws[laws['CELEX'] == celex_id]['Act_type'].iloc[0]}
        'treaty': {laws[laws['CELEX'] == celex_id]['Treaty'].iloc[0]}

        full_doc :

        {laws[laws['CELEX'] == celex_id]['act_raw_text'].iloc[0]}
{"=="*15} "END OF DOC" {"=="*15}
        """

    return full_doc_info    



In [ ]:
display(Markdown(search_docs("driving without license penalty"))) 

In [23]:
laws[laws['CELEX'] == '32015L0413']['CELEX'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Status'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Act_type'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Treaty'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['act_raw_text'].iloc[0]


"13.3.2015 EN Official Journal of the European Union L 68/9 DIRECTIVE (EU) 2015/413 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 11 March 2015 facilitating cross-border exchange of information on road-safety-related traffic offences (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 91(1)(c) thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the ordinary legislative procedure (2), Whereas: (1) Improving road safety is a prime objective of the Union's transport policy. The Union is pursuing a policy to improve road safety with the objective of reducing fatalities, injuries and material damage. An important e

In [ ]:
def search_docs(query):
    celex_ids : list[str,str]
    full_doc_info : list[str]

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list[Any](set(celex_ids))    

    
    full_doc_info = f""" Doc{i}:\n 
    
      {laws[laws['CELEX'] == celex_id]['act_raw_text'].iloc[0]} 

""" 

    return full_doc_info    



In [ ]:
search_docs("Drug dealing Sentences")

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("Drug dealing Sentences")

retrived = []

for i , doc in enumerate (docs):
    retrived.append([f"Doc{i}:"  doc.metadata['status','treaty','act_type','celex']])


[Document(metadata={'subject_matter': 'sources and branches of the law;  European Union law;  justice;  criminal law', 'status': 'In Force', 'legal_basis': '12002M031; 12002M034', 'additional_info': 'CNS 2001/0114', 'chunk_number': 1, 'eurovoc': 'penal code; Community law - national law; criminal procedure; penalty; drug traffic', 'treaty': 'TEU (1992)', 'act_type': 'Decision_FRAMW', 'cites': 'joint_action/1997/396; 31999Y0123%2801%29; joint_action/1998/733', 'celex': '32004F0757', 'authors': 'European Council', 'act_name': 'Council Framework Decision 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the field of illicit drug trafficking', 'document_length': 13663, 'total_chunks': 6}, page_content="11.11.2004 EN Official Journal of the European Union L 335/8 COUNCIL FRAMEWORK DECISION 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the 

In [ ]:
retrived = []

for doc in docs:
    retrived.append({doc.metadata['status','treaty']})

In [45]:
docs[0].metadata['celex']

'32004F0757'

In [ ]:
def get_celex_ids(docs):
    celex_ids = []
    for i in docs:
        celex_ids.append(docs[i].metadata['celex'])

    celex_ids = list(set(celex_ids))

    return celex_ids    

In [1]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI."),
    ("human", "Answer this question: {question}")
])

formatted = prompt.invoke({"question": "What is AI?"})
print(formatted)

messages=[SystemMessage(content='You are a helpful AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Answer this question: What is AI?', additional_kwargs={}, response_metadata={})]


In [9]:
laws[laws['CELEX'] == '31997R2046']['act_raw_text'].iloc[0]   

"Avis juridique important|31997R2046Council Regulation (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addiction Official Journal L 287 , 21/10/1997 P. 0001 - 0005COUNCIL REGULATION (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addictionTHE COUNCIL OF THE EUROPEAN UNION,Having regard to the Treaty establishing the European Community, and in particular Article 130w thereof,Having regard to the proposal from the Commission (1),Acting in accordance with the procedure laid down in Article 189c of the Treaty (2),Whereas the impact on the structures of a developing society of an economy based on the production of drugs, or which derives a substantial revenue from them, undermines a country's smooth integration into the world economy;Whereas the breakdown of social structures in developing countries due to drug consumption and the related industry is detrimental to sustainable social de

In [11]:
def get_full_docs_celex(celex_ids):
    full_doc_info = """ """

    # open weaviate client
    weaviate_client.connect()

    # Remove duplicates from celex_ids list
    celex_ids = list(set(celex_ids))    

    for i , celex_id in enumerate(celex_ids):
        f_doc_meta = get_full_doc_weaviate(celex_id)

        full_doc_info += f"""

        doc {i} :

        'celex': {f_doc_meta['celex']}
        'status': {f_doc_meta['status']}
        'act_type': {f_doc_meta['act_type']}
        'treaty': {f_doc_meta['treaty']}

        full_doc :

        {f_doc_meta['full_doc']}
{"=="*15} "END OF DOC" {"=="*15}
        """
        # for debugging purpose
        if len(full_doc_info) > 5:
            print("docs found and retrieved successfully")
    print(len(full_doc_info))

    #open weaviate client
    weaviate_client.close()      

    return full_doc_info

In [24]:
celex_list = ['32018R1725', '32018D1962']

In [ ]:
display(Markdown(get_full_docs_celex(celex_list)))